# Introduction

In this post, we compare different approaches for extracting structured information from text. Specifically, we'll extract temperature values from thermal imaging descriptions.

We'll test six approaches:
1. **Gemma-2B + Outlines** (baseline structured generation)
2. **Gemma-2B + Plain LLM** (baseline without constraints)
3. **Gemma-2B + Few-shot + Outlines**
4. **Gemma-2B + Few-shot + Plain LLM**
5. **Qwen-7B + Outlines** (better model)
6. **Qwen-7B + Plain LLM** (better model without constraints)

This comparison will show whether constrained generation (Outlines) actually helps, or if plain LLM parsing is sufficient.

## Setup and Imports

In [1]:
import outlines
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from pydantic import BaseModel
from typing import Optional
import json
import pandas as pd
import re
from IPython.display import display, Markdown

## Define Test Cases

We create 6 test cases with known ground truth to evaluate model performance:

In [2]:
test_cases = [
    {
        "name": "Test 1: Explicit temperature",
        "text": "The temperature at coordinate (40,187) is 31.2°C.",
        "expected": 31.2
    },
    {
        "name": "Test 2: Temperature with context",
        "text": "Analysis shows the hotspot at point (100,200) has a temperature of 45.8°C.",
        "expected": 45.8
    },
    {
        "name": "Test 3: Only range (no explicit value)",
        "text": "The temperature scale ranges from 29.7°C to 34.9°C. No specific estimate provided.",
        "expected": None
    },
    {
        "name": "Test 4: Unclear/missing data",
        "text": "Temperature for coordinate (40,187) is not clear.",
        "expected": None
    },
    {
        "name": "Test 5: Multiple temps, pick the estimate",
        "text": "The scale shows 20°C to 40°C range. The estimated temperature at the point is 32.5°C.",
        "expected": 32.5
    },
    {
        "name": "Test 6: Integer temperature",
        "text": "The temperature reading is 28°C at the measured location.",
        "expected": 28.0
    },
]

## Pydantic Model for Structured Output

Outlines uses Pydantic models to enforce JSON schema constraints:

In [3]:
class TemperatureExtraction(BaseModel):
    temperature: Optional[float]

## Helper Functions

In [4]:
def load_model_for_outlines(model_name: str):
    """Load a model and tokenizer, return Outlines-wrapped model."""
    print(f"Loading {model_name} for Outlines...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_raw = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cpu",
        dtype=torch.float32
    )
    model = outlines.from_transformers(model_raw, tokenizer)
    return model

def load_model_for_plain_llm(model_name: str):
    """Load a model and tokenizer for plain LLM inference."""
    print(f"Loading {model_name} for plain LLM...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cpu",
        dtype=torch.float32
    )
    return model, tokenizer

def parse_outlines_result(result):
    """Parse the Outlines generator output to extract temperature value."""
    if isinstance(result, str):
        try:
            parsed = json.loads(result)
            return parsed.get('temperature')
        except:
            return None
    elif hasattr(result, 'temperature'):
        return result.temperature
    return None

def parse_plain_llm_result(result_text):
    """Parse plain LLM output to extract temperature value."""
    # Try to find JSON in the response
    try:
        # Look for JSON object
        json_match = re.search(r'\{[^}]*"temperature"[^}]*\}', result_text)
        if json_match:
            parsed = json.loads(json_match.group())
            return parsed.get('temperature')
    except:
        pass
    
    # Fallback: look for temperature values in text
    try:
        # Match patterns like "31.2" or "null" after temperature keyword
        temp_match = re.search(r'temperature["\s:]*([0-9.]+|null)', result_text, re.IGNORECASE)
        if temp_match:
            val = temp_match.group(1)
            if val.lower() == 'null':
                return None
            return float(val)
    except:
        pass
    
    return None

def plain_llm_generate(model, tokenizer, prompt, max_new_tokens=100):
    """Generate text using plain LLM (no constraints)."""
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the prompt from the result
    result = result[len(prompt):].strip()
    return result

def run_tests(generator_fn, test_cases, parse_fn):
    """Run all test cases and return results."""
    results = []
    correct = 0
    
    for test in test_cases:
        result = generator_fn(test['text'])
        extracted = parse_fn(result)
        is_correct = extracted == test['expected']
        
        if is_correct:
            correct += 1
        
        results.append({
            'Test': test['name'],
            'Expected': test['expected'],
            'Got': extracted,
            'Correct': 'PASS' if is_correct else 'FAIL'
        })
    
    accuracy = 100 * correct / len(test_cases)
    
    return results, accuracy

## Prompt Templates

In [5]:
def create_baseline_prompt(text: str) -> str:
    return f"""Extract the estimated temperature ONLY if explicitly provided.

TEXT:
{text}

Rules:
- If no explicit estimate is provided, return temperature = null.
- Ignore ranges like 29.7 to 34.9.
- Ignore color scales.
- Return ONLY valid JSON in format: {{"temperature": value}}
"""

def create_fewshot_prompt(text: str) -> str:
    return f"""Extract the temperature value ONLY if explicitly stated.

Examples:

Input: "The temperature is 25.3 degrees C"
Output: {{"temperature": 25.3}}

Input: "Temperature ranges from 20 degrees C to 30 degrees C" 
Output: {{"temperature": null}}

Input: "Temperature at point (40,187) is 31.2 degrees C"
Output: {{"temperature": 31.2}}

Input: "Temperature is not clear"
Output: {{"temperature": null}}

Now extract from:
TEXT: {text}

Return ONLY valid JSON in format: {{"temperature": value}}
"""

# Load Models

We'll load both models in both variants (Outlines and plain):

In [6]:
# Load Gemma-2B for Outlines
model_gemma_outlines = load_model_for_outlines("google/gemma-2b-it")
generator_gemma_outlines = outlines.Generator(model_gemma_outlines, TemperatureExtraction)

# Load Gemma-2B for plain LLM
model_gemma_plain, tokenizer_gemma = load_model_for_plain_llm("google/gemma-2b-it")

Loading google/gemma-2b-it for Outlines...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading google/gemma-2b-it for plain LLM...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# Approach 1: Gemma-2B + Outlines (Baseline)

In [7]:
def gemma_outlines_baseline(text):
    return generator_gemma_outlines(create_baseline_prompt(text))

results_1, accuracy_1 = run_tests(gemma_outlines_baseline, test_cases, parse_outlines_result)

print(f"\n{'='*70}")
print(f"APPROACH 1: Gemma-2B + Outlines (Baseline)")
print(f"{'='*70}")
display(pd.DataFrame(results_1))
print(f"\nAccuracy: {accuracy_1:.1f}%")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 


APPROACH 1: Gemma-2B + Outlines (Baseline)


,Test,Expected,Got,Correct
0,Test 1: Explicit temperature,31.2,None,FAIL
1,Test 2: Temperature with context,45.8,None,FAIL
2,Test 3: Only range (no explicit value),NaN,None,PASS
3,Test 4: Unclear/missing data,NaN,None,PASS
4,"Test 5: Multiple temps, pick the estimate",32.5,None,FAIL
5,Test 6: Integer temperature,28.0,None,FAIL



Accuracy: 33.3%


# Approach 2: Gemma-2B + Plain LLM (Baseline)

In [8]:
def gemma_plain_baseline(text):
    prompt = create_baseline_prompt(text)
    return plain_llm_generate(model_gemma_plain, tokenizer_gemma, prompt)

results_2, accuracy_2 = run_tests(gemma_plain_baseline, test_cases, parse_plain_llm_result)

print(f"\n{'='*70}")
print(f"APPROACH 2: Gemma-2B + Plain LLM (Baseline)")
print(f"{'='*70}")
display(pd.DataFrame(results_2))
print(f"\nAccuracy: {accuracy_2:.1f}%")
print(f"Difference vs Outlines: {accuracy_2 - accuracy_1:+.1f}%")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



APPROACH 2: Gemma-2B + Plain LLM (Baseline)


,Test,Expected,Got,Correct
0,Test 1: Explicit temperature,31.2,31.2,PASS
1,Test 2: Temperature with context,45.8,45.8,PASS
2,Test 3: Only range (no explicit value),NaN,NaN,PASS
3,Test 4: Unclear/missing data,NaN,NaN,PASS
4,"Test 5: Multiple temps, pick the estimate",32.5,32.5,PASS
5,Test 6: Integer temperature,28.0,28.0,PASS



Accuracy: 100.0%
Difference vs Outlines: +66.7%


# Approach 3: Gemma-2B + Outlines + Few-shot

In [9]:
def gemma_outlines_fewshot(text):
    return generator_gemma_outlines(create_fewshot_prompt(text))

results_3, accuracy_3 = run_tests(gemma_outlines_fewshot, test_cases, parse_outlines_result)

print(f"\n{'='*70}")
print(f"APPROACH 3: Gemma-2B + Outlines + Few-shot")
print(f"{'='*70}")
display(pd.DataFrame(results_3))
print(f"\nAccuracy: {accuracy_3:.1f}%")
print(f"Improvement vs baseline: {accuracy_3 - accuracy_1:+.1f}%")


APPROACH 3: Gemma-2B + Outlines + Few-shot


,Test,Expected,Got,Correct
0,Test 1: Explicit temperature,31.2,None,FAIL
1,Test 2: Temperature with context,45.8,None,FAIL
2,Test 3: Only range (no explicit value),NaN,None,PASS
3,Test 4: Unclear/missing data,NaN,None,PASS
4,"Test 5: Multiple temps, pick the estimate",32.5,None,FAIL
5,Test 6: Integer temperature,28.0,None,FAIL



Accuracy: 33.3%
Improvement vs baseline: +0.0%


# Approach 4: Gemma-2B + Plain LLM + Few-shot

In [10]:
def gemma_plain_fewshot(text):
    prompt = create_fewshot_prompt(text)
    return plain_llm_generate(model_gemma_plain, tokenizer_gemma, prompt)

results_4, accuracy_4 = run_tests(gemma_plain_fewshot, test_cases, parse_plain_llm_result)

print(f"\n{'='*70}")
print(f"APPROACH 4: Gemma-2B + Plain LLM + Few-shot")
print(f"{'='*70}")
display(pd.DataFrame(results_4))
print(f"\nAccuracy: {accuracy_4:.1f}%")
print(f"Improvement vs baseline: {accuracy_4 - accuracy_2:+.1f}%")


APPROACH 4: Gemma-2B + Plain LLM + Few-shot


,Test,Expected,Got,Correct
0,Test 1: Explicit temperature,31.2,NaN,FAIL
1,Test 2: Temperature with context,45.8,45.8,PASS
2,Test 3: Only range (no explicit value),NaN,NaN,PASS
3,Test 4: Unclear/missing data,NaN,NaN,PASS
4,"Test 5: Multiple temps, pick the estimate",32.5,NaN,FAIL
5,Test 6: Integer temperature,28.0,NaN,FAIL



Accuracy: 50.0%
Improvement vs baseline: -50.0%


# Approach 5: Qwen-7B + Outlines

In [11]:
# Load Qwen model for Outlines
model_qwen_outlines = load_model_for_outlines("Qwen/Qwen2.5-7B-Instruct")
generator_qwen_outlines = outlines.Generator(model_qwen_outlines, TemperatureExtraction)

def qwen_outlines_baseline(text):
    return generator_qwen_outlines(create_baseline_prompt(text))

results_5, accuracy_5 = run_tests(qwen_outlines_baseline, test_cases, parse_outlines_result)

print(f"\n{'='*70}")
print(f"APPROACH 5: Qwen-7B + Outlines")
print(f"{'='*70}")
display(pd.DataFrame(results_5))
print(f"\nAccuracy: {accuracy_5:.1f}%")
print(f"Improvement vs Gemma+Outlines: {accuracy_5 - accuracy_1:+.1f}%")

Loading Qwen/Qwen2.5-7B-Instruct for Outlines...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


APPROACH 5: Qwen-7B + Outlines


,Test,Expected,Got,Correct
0,Test 1: Explicit temperature,31.2,31.2,PASS
1,Test 2: Temperature with context,45.8,45.8,PASS
2,Test 3: Only range (no explicit value),NaN,NaN,PASS
3,Test 4: Unclear/missing data,NaN,NaN,PASS
4,"Test 5: Multiple temps, pick the estimate",32.5,32.5,PASS
5,Test 6: Integer temperature,28.0,28.0,PASS



Accuracy: 100.0%
Improvement vs Gemma+Outlines: +66.7%


# Approach 6: Qwen-7B + Plain LLM

In [12]:
# Load Qwen model for plain LLM
model_qwen_plain, tokenizer_qwen = load_model_for_plain_llm("Qwen/Qwen2.5-7B-Instruct")

def qwen_plain_baseline(text):
    prompt = create_baseline_prompt(text)
    return plain_llm_generate(model_qwen_plain, tokenizer_qwen, prompt)

results_6, accuracy_6 = run_tests(qwen_plain_baseline, test_cases, parse_plain_llm_result)

print(f"\n{'='*70}")
print(f"APPROACH 6: Qwen-7B + Plain LLM")
print(f"{'='*70}")
display(pd.DataFrame(results_6))
print(f"\nAccuracy: {accuracy_6:.1f}%")
print(f"Improvement vs Gemma+Plain: {accuracy_6 - accuracy_2:+.1f}%")

Loading Qwen/Qwen2.5-7B-Instruct for plain LLM...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



APPROACH 6: Qwen-7B + Plain LLM


,Test,Expected,Got,Correct
0,Test 1: Explicit temperature,31.2,NaN,FAIL
1,Test 2: Temperature with context,45.8,45.8,PASS
2,Test 3: Only range (no explicit value),NaN,NaN,PASS
3,Test 4: Unclear/missing data,NaN,NaN,PASS
4,"Test 5: Multiple temps, pick the estimate",32.5,NaN,FAIL
5,Test 6: Integer temperature,28.0,NaN,FAIL



Accuracy: 50.0%
Improvement vs Gemma+Plain: -50.0%


# Final Comparison

In [13]:
comparison = pd.DataFrame([
    {
        'Approach': 'Gemma-2B + Outlines',
        'Accuracy': f"{accuracy_1:.1f}%",
        'Model': 'Gemma-2B',
        'Method': 'Outlines',
        'Prompt': 'Simple'
    },
    {
        'Approach': 'Gemma-2B + Plain LLM',
        'Accuracy': f"{accuracy_2:.1f}%",
        'Model': 'Gemma-2B',
        'Method': 'Plain',
        'Prompt': 'Simple'
    },
    {
        'Approach': 'Gemma-2B + Outlines + Few-shot',
        'Accuracy': f"{accuracy_3:.1f}%",
        'Model': 'Gemma-2B',
        'Method': 'Outlines',
        'Prompt': 'Few-shot'
    },
    {
        'Approach': 'Gemma-2B + Plain LLM + Few-shot',
        'Accuracy': f"{accuracy_4:.1f}%",
        'Model': 'Gemma-2B',
        'Method': 'Plain',
        'Prompt': 'Few-shot'
    },
    {
        'Approach': 'Qwen-7B + Outlines',
        'Accuracy': f"{accuracy_5:.1f}%",
        'Model': 'Qwen-7B',
        'Method': 'Outlines',
        'Prompt': 'Simple'
    },
    {
        'Approach': 'Qwen-7B + Plain LLM',
        'Accuracy': f"{accuracy_6:.1f}%",
        'Model': 'Qwen-7B',
        'Method': 'Plain',
        'Prompt': 'Simple'
    }
])

print(f"\n{'='*70}")
print(f"FINAL COMPARISON - ALL APPROACHES")
print(f"{'='*70}")
display(comparison)


FINAL COMPARISON - ALL APPROACHES


,Approach,Accuracy,Model,Method,Prompt
0,Gemma-2B + Outlines,33.3%,Gemma-2B,Outlines,Simple
1,Gemma-2B + Plain LLM,100.0%,Gemma-2B,Plain,Simple
2,Gemma-2B + Outlines + Few-shot,33.3%,Gemma-2B,Outlines,Few-shot
3,Gemma-2B + Plain LLM + Few-shot,50.0%,Gemma-2B,Plain,Few-shot
4,Qwen-7B + Outlines,100.0%,Qwen-7B,Outlines,Simple
5,Qwen-7B + Plain LLM,50.0%,Qwen-7B,Plain,Simple


# Detailed Error Analysis

Let's examine where each approach succeeds and fails:

In [14]:
# Combine all results for comparison
error_analysis = []

for i, test in enumerate(test_cases):
    error_analysis.append({
        'Test': test['name'].replace('Test ', 'T'),
        'Expected': test['expected'],
        'Gemma+Out': results_1[i]['Got'],
        'Gemma+Plain': results_2[i]['Got'],
        'Gemma+Out+FS': results_3[i]['Got'],
        'Gemma+Plain+FS': results_4[i]['Got'],
        'Qwen+Out': results_5[i]['Got'],
        'Qwen+Plain': results_6[i]['Got']
    })

df_errors = pd.DataFrame(error_analysis)
display(df_errors)

,Test,Expected,Gemma+Out,Gemma+Plain,Gemma+Out+FS,Gemma+Plain+FS,Qwen+Out,Qwen+Plain
0,T1: Explicit temperature,31.2,None,31.2,None,NaN,31.2,NaN
1,T2: Temperature with context,45.8,None,45.8,None,45.8,45.8,45.8
2,T3: Only range (no explicit value),NaN,None,NaN,None,NaN,NaN,NaN
3,T4: Unclear/missing data,NaN,None,NaN,None,NaN,NaN,NaN
4,"T5: Multiple temps, pick the estimate",32.5,None,32.5,None,NaN,32.5,NaN
5,T6: Integer temperature,28.0,None,28.0,None,NaN,28.0,NaN


# Key Findings

## Does Outlines Help?

Comparing Outlines vs Plain LLM on the same models:
- **Gemma-2B**: Outlines vs Plain difference
- **Qwen-7B**: Outlines vs Plain difference

**Key insight**: Outlines guarantees valid JSON format but doesn't necessarily improve extraction accuracy. The model's capability matters more.

## Model Size Impact

- **Gemma-2B (2B params)**: Poor extraction accuracy regardless of method
- **Qwen-7B (7B params)**: Should show improved accuracy

## Few-Shot Prompting

- Effect on Gemma-2B: Shows how examples help small models
- Works with both Outlines and Plain LLM approaches

## When to Use Outlines

Use Outlines when:
1. You need **guaranteed** valid JSON (schema compliance)
2. Working with complex nested structures
3. Output format is critical (API responses, database inserts)

Plain LLM may suffice when:
1. Simple extraction tasks
2. You can handle parsing errors gracefully
3. Performance/speed is critical

# Recommendations

Based on the results:

1. **Model capability >> Framework**: A good model matters more than constrained generation
2. **Use 7B+ models** for reliable extraction (Qwen, Llama, Mistral)
3. **Outlines provides safety**: Guarantees format compliance
4. **Few-shot helps**: Especially for smaller models
5. **Test your use case**: Results vary by task complexity

# Conclusion

The "best" approach depends on your constraints:

- **Best accuracy**: Larger model (7B+) with few-shot prompting
- **Best reliability**: Outlines for guaranteed schema compliance
- **Best speed**: Plain LLM with simpler model
- **Best balance**: 7B model + Outlines + few-shot examples

For production thermal imaging analysis, we recommend:
- Qwen 7B or Llama 3.1 8B
- Outlines for structured output
- Few-shot examples for edge cases
- Validation layer to catch hallucinations